In [1]:
import sys
sys.path.append("..")
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from src.features import make_features

df_clean = pd.read_parquet("../data/processed/electricity_clean.parquet")
temp = pd.read_parquet("../data/processed/air_temperature.parquet")
train_buildings = pd.read_csv("../configs/train_buildings.csv")["building"].tolist()

rng = np.random.default_rng(0)
sample = rng.choice(train_buildings, size=100, replace=False)
data = pd.concat([make_features(b, df_clean, temp) for b in sample])

FEATS = ["hour", "weekday", "lag24_n", "lag168_n", "temp"]
train = data[data.index.year == 2016].dropna(subset=["load_n"])
test = data[data.index.year == 2017].dropna(subset=["load_n"])
print(len(train), "train rows,", len(test), "test rows")

845554 train rows, 864321 test rows


In [2]:
models = [HistGradientBoostingRegressor(random_state=s).fit(train[FEATS], train["load_n"])
          for s in range(5)]
print("trees per seed:", [m.n_iter_ for m in models])

P = np.column_stack([m.predict(test[FEATS]) for m in models])
diff = np.abs(P - P[:, [0]])
print("mean |difference| vs seed 0:", diff.mean(axis=0).round(4))
print("max  |difference| vs seed 0:", diff.max(axis=0).round(4))

trees per seed: [100, 100, 100, 100, 100]
mean |difference| vs seed 0: [0.     0.0093 0.0093 0.0095 0.0092]
max  |difference| vs seed 0: [0.     3.9397 7.7285 4.1412 5.8092]


In [3]:
from itertools import combinations
from math import factorial

def exact_shap(model, X, background):
    """Exact interventional Shapley values. X: (n, 5) hours to explain, background: (b, 5)."""
    n, p = X.shape
    b = len(background)
    v = {}
    for size in range(p + 1):
        for S in combinations(range(p), size):
            Z = np.repeat(background[None, :, :], n, axis=0)
            Z[:, :, list(S)] = X[:, None, list(S)]
            Zdf = pd.DataFrame(Z.reshape(n * b, p), columns=FEATS)
            v[S] = model.predict(Zdf).reshape(n, b).mean(axis=1)
    phi = np.zeros((n, p))
    for j in range(p):
        others = [k for k in range(p) if k != j]
        for size in range(p):
            w = factorial(size) * factorial(p - size - 1) / factorial(p)
            for S in combinations(others, size):
                phi[:, j] += w * (v[tuple(sorted(S + (j,)))] - v[S])
    return phi, v[()]

X_explain = test[FEATS].sample(1000, random_state=0).to_numpy()
background = train[FEATS].sample(100, random_state=0).to_numpy()

phi, base = exact_shap(models[0], X_explain, background)
pred = models[0].predict(pd.DataFrame(X_explain, columns=FEATS))
print("additivity error (should be ~0):", np.abs(phi.sum(axis=1) + base - pred).max())
print(pd.Series(np.abs(phi).mean(axis=0), index=FEATS).sort_values(ascending=False).round(3))

additivity error (should be ~0): 1.7763568394002505e-15
lag24_n     0.113
lag168_n    0.052
weekday     0.033
hour        0.014
temp        0.011
dtype: float64


In [4]:
worst = np.argsort(diff.max(axis=1))[-8:]
show = test.iloc[worst][["building", "hour", "weekday", "lag24_n", "lag168_n", "temp", "load_n"]].copy()
for s in range(5):
    show[f"seed{s}"] = P[worst, s]
print(show.round(2).to_string())

                                  building  hour  weekday  lag24_n  lag168_n  temp  load_n  seed0  seed1  seed2  seed3  seed4
2017-07-28 07:00:00   Gator_public_Natasha     7        4     2.30      1.12  26.1    2.82   1.72   1.72   5.00   1.71   1.73
2017-10-10 12:00:00     Gator_public_Jayme    12        1      NaN       NaN  31.7    1.18   6.68   5.20   4.65   2.62   4.40
2017-10-10 14:00:00     Gator_public_Jayme    14        1      NaN       NaN  31.7    1.17   6.76   4.92   5.14   2.62   4.40
2017-09-25 20:00:00       Gator_public_Noe    20        0      NaN       NaN  26.1    0.90  16.96  20.62  19.27  17.78  21.99
2017-09-11 14:00:00  Bear_education_Santos    14        0      NaN       NaN  28.9    1.12  17.79  17.31  19.80  16.69  22.95
2017-09-25 09:00:00       Gator_public_Noe     9        0      NaN       NaN  27.2    0.91  16.61  20.55  20.76  19.33  22.39
2017-09-25 10:00:00       Gator_public_Noe    10        0      NaN       NaN  28.9    0.91  18.25  17.91  19.61  17.22

In [5]:
from scipy.stats import spearmanr

phis = [exact_shap(m, X_explain, background)[0] for m in models]
imp = pd.DataFrame([np.abs(p).mean(axis=0) for p in phis], columns=FEATS,
                   index=[f"seed {s}" for s in range(5)])
print(imp.round(4))

pairs = list(combinations(range(5), 2))
rho = [spearmanr(imp.iloc[a], imp.iloc[b])[0] for a, b in pairs]
top2 = [set(imp.iloc[a].nlargest(2).index) == set(imp.iloc[b].nlargest(2).index)
        for a, b in pairs]

def cosine(A, B):
    return (A * B).sum(axis=1) / (np.linalg.norm(A, axis=1) * np.linalg.norm(B, axis=1))

cos = np.concatenate([cosine(phis[a], phis[b]) for a, b in pairs])

print(f"Spearman: mean {np.mean(rho):.3f}, min {np.min(rho):.3f}")
print(f"Top-2 agreement: {np.mean(top2):.2f} of {len(pairs)} pairs")
print(f"Local cosine: median {np.median(cos):.4f}, 5th percentile {np.percentile(cos, 5):.4f}")

          hour  weekday  lag24_n  lag168_n    temp
seed 0  0.0137   0.0326   0.1129    0.0518  0.0109
seed 1  0.0129   0.0328   0.1125    0.0514  0.0106
seed 2  0.0129   0.0328   0.1123    0.0517  0.0106
seed 3  0.0133   0.0321   0.1129    0.0518  0.0106
seed 4  0.0137   0.0329   0.1121    0.0508  0.0103
Spearman: mean 1.000, min 1.000
Top-2 agreement: 1.00 of 10 pairs
Local cosine: median 0.9989, 5th percentile 0.9855


In [6]:
both_nan = (test["lag24_n"].isna() & test["lag168_n"].isna()).to_numpy()
print(both_nan.sum(), "test hours with both lags missing, in",
      test.loc[both_nan, "building"].nunique(), "buildings")
print((train["lag24_n"].isna() & train["lag168_n"].isna()).sum(), "such hours in training")

err = np.abs(test["load_n"].to_numpy() - P[:, 0])
print("mean |error| (normalised), both lags missing:", err[both_nan].mean().round(3))
print("mean |error| (normalised), all other hours:  ", err[~both_nan].mean().round(3))

375 test hours with both lags missing, in 22 buildings
600 such hours in training
mean |error| (normalised), both lags missing: 1.666
mean |error| (normalised), all other hours:   0.113


In [7]:
def stability(models):
    phis = [exact_shap(m, X_explain, background)[0] for m in models]
    imp = pd.DataFrame([np.abs(p).mean(axis=0) for p in phis], columns=FEATS,
                       index=[f"model {i}" for i in range(len(models))])
    pairs = list(combinations(range(len(models)), 2))
    rho = [spearmanr(imp.iloc[a], imp.iloc[b])[0] for a, b in pairs]
    top2 = [set(imp.iloc[a].nlargest(2).index) == set(imp.iloc[b].nlargest(2).index)
            for a, b in pairs]
    cos_pairs = np.vstack([cosine(phis[a], phis[b]) for a, b in pairs])
    print(imp.round(4))
    print(f"Spearman: mean {np.mean(rho):.3f}, min {np.min(rho):.3f}")
    print(f"Top-2 agreement: {np.mean(top2):.2f} of {len(pairs)} pairs")
    print(f"Local cosine: median {np.median(cos_pairs):.4f}, "
          f"5th percentile {np.percentile(cos_pairs, 5):.4f}")
    return phis, imp, cos_pairs.min(axis=0)

rng_b = np.random.default_rng(1)
by_building = {b: g for b, g in train.groupby("building")}
boot_models = []
for i in range(5):
    picked = rng_b.choice(sample, size=len(sample), replace=True)
    tr = pd.concat([by_building[b] for b in picked])
    boot_models.append(HistGradientBoostingRegressor(random_state=0).fit(tr[FEATS], tr["load_n"]))
    print("bootstrap model", i, "done:", len(set(picked)), "different buildings", flush=True)

phis_b, imp_b, worst_cos = stability(boot_models)

low = np.argsort(worst_cos)[:8]
print(pd.DataFrame(X_explain[low], columns=FEATS).assign(min_cosine=worst_cos[low]).round(3))

bootstrap model 0 done: 64 different buildings
bootstrap model 1 done: 67 different buildings
bootstrap model 2 done: 65 different buildings
bootstrap model 3 done: 62 different buildings
bootstrap model 4 done: 67 different buildings
           hour  weekday  lag24_n  lag168_n    temp
model 0  0.0145   0.0301   0.1136    0.0492  0.0095
model 1  0.0148   0.0357   0.1133    0.0492  0.0180
model 2  0.0188   0.0381   0.1078    0.0474  0.0109
model 3  0.0114   0.0331   0.1139    0.0510  0.0125
model 4  0.0189   0.0348   0.1144    0.0508  0.0128
Spearman: mean 0.940, min 0.900
Top-2 agreement: 1.00 of 10 pairs
Local cosine: median 0.9907, 5th percentile 0.8963
   hour  weekday  lag24_n  lag168_n  temp  min_cosine
0   4.0      4.0    1.032     0.970  23.3      -0.413
1  10.0      0.0    0.903     0.202  30.6      -0.348
2   0.0      1.0      NaN     0.821  15.6       0.114
3  23.0      2.0    0.527     0.414  28.9       0.226
4   9.0      4.0      NaN     0.797  25.6       0.374
5   1.0     

In [8]:
size = np.linalg.norm(np.mean(phis_b, axis=0), axis=1)
small = size < np.percentile(size, 20)
print("median min-cosine, smallest 20% of explanations:", np.median(worst_cos[small]).round(3))
print("median min-cosine, the other 80%:              ", np.median(worst_cos[~small]).round(3))

lag_nan = np.isnan(X_explain[:, 2]) | np.isnan(X_explain[:, 3])
print(lag_nan.sum(), "explained hours with a missing lag, median min-cosine:",
      np.median(worst_cos[lag_nan]).round(3))

def total_abs_diff(phis):
    pairs = list(combinations(range(len(phis)), 2))
    return np.mean([np.abs(phis[a] - phis[b]).sum(axis=1) for a, b in pairs], axis=0)

for name, ph in [("seeds", phis), ("bootstrap", phis_b)]:
    d = total_abs_diff(ph)
    print(f"{name}: total |SHAP difference| per hour, median {np.median(d):.4f}, "
          f"95th percentile {np.percentile(d, 95):.4f}")

median min-cosine, smallest 20% of explanations: 0.915
median min-cosine, the other 80%:               0.981
15 explained hours with a missing lag, median min-cosine: 0.875
seeds: total |SHAP difference| per hour, median 0.0106, 95th percentile 0.0384
bootstrap: total |SHAP difference| per hour, median 0.0300, 95th percentile 0.1253


In [9]:
GROUPS = {"past load": [2, 3], "calendar": [0, 1], "temperature": [4]}

def group_shap(model, X, background, groups):
    names = list(groups)
    g, (n, p), b = len(names), X.shape, len(background)
    v = {}
    for size in range(g + 1):
        for G in combinations(range(g), size):
            cols = [c for k in G for c in groups[names[k]]]
            Z = np.repeat(background[None, :, :], n, axis=0)
            Z[:, :, cols] = X[:, None, cols]
            Zdf = pd.DataFrame(Z.reshape(n * b, p), columns=FEATS)
            v[G] = model.predict(Zdf).reshape(n, b).mean(axis=1)
    phi = np.zeros((n, g))
    for j in range(g):
        others = [k for k in range(g) if k != j]
        for size in range(g):
            w = factorial(size) * factorial(g - size - 1) / factorial(g)
            for G in combinations(others, size):
                phi[:, j] += w * (v[tuple(sorted(G + (j,)))] - v[G])
    return phi, v[()]

gphis = []
for m in boot_models:
    ph, base_g = group_shap(m, X_explain, background, GROUPS)
    pred_m = m.predict(pd.DataFrame(X_explain, columns=FEATS))
    assert np.abs(ph.sum(axis=1) + base_g - pred_m).max() < 1e-9   # additivity check
    gphis.append(ph)

gimp = pd.DataFrame([np.abs(ph).mean(axis=0) for ph in gphis], columns=list(GROUPS),
                    index=[f"model {i}" for i in range(5)])
print(gimp.round(4))

pairs = list(combinations(range(5), 2))
gcos = np.vstack([cosine(gphis[a], gphis[b]) for a, b in pairs])
print(f"Group cosine: median {np.median(gcos):.4f}, 5th percentile {np.percentile(gcos, 5):.4f}")
d = total_abs_diff(gphis)
print(f"Group total |SHAP difference|: median {np.median(d):.4f}, 95th percentile {np.percentile(d, 95):.4f}")

share = [np.abs(ph[:, 2]).mean() / (np.abs(ph[:, 2]).mean() + np.abs(ph[:, 3]).mean()) for ph in phis_b]
print("share of past-load credit going to lag24_n, per bootstrap model:", np.round(share, 3))

         past load  calendar  temperature
model 0     0.1476    0.0328       0.0093
model 1     0.1462    0.0378       0.0183
model 2     0.1413    0.0413       0.0112
model 3     0.1481    0.0337       0.0126
model 4     0.1489    0.0397       0.0127
Group cosine: median 0.9956, 5th percentile 0.8351
Group total |SHAP difference|: median 0.0231, 95th percentile 0.1099
share of past-load credit going to lag24_n, per bootstrap model: [0.698 0.697 0.694 0.691 0.692]


In [10]:
windows = {"Jan–Jun": (1, 6), "Mar–Aug": (3, 8), "May–Oct": (5, 10), "Jul–Dec": (7, 12)}
train_month = train.index.month
period_models = []
for name, (first, last) in windows.items():
    tr = train[(train_month >= first) & (train_month <= last)]
    m = HistGradientBoostingRegressor(random_state=0).fit(tr[FEATS], tr["load_n"])
    mae = np.abs(test["load_n"].to_numpy() - m.predict(test[FEATS])).mean()
    print(f"{name}: {len(tr)} training rows, test MAE (normalised) {mae:.4f}", flush=True)
    period_models.append(m)

phis_p, imp_p, worst_cos_p = stability(period_models)
d = total_abs_diff(phis_p)
print(f"Total |SHAP difference|: median {np.median(d):.4f}, 95th percentile {np.percentile(d, 95):.4f}")

gimp_p = pd.DataFrame(
    [np.abs(group_shap(m, X_explain, background, GROUPS)[0]).mean(axis=0) for m in period_models],
    columns=list(GROUPS), index=list(windows))
print(gimp_p.round(4))

Jan–Jun: 414950 training rows, test MAE (normalised) 0.1143
Mar–Aug: 426016 training rows, test MAE (normalised) 0.1153
May–Oct: 432118 training rows, test MAE (normalised) 0.1153
Jul–Dec: 430604 training rows, test MAE (normalised) 0.1141
           hour  weekday  lag24_n  lag168_n    temp
model 0  0.0174   0.0328   0.1081    0.0517  0.0107
model 1  0.0134   0.0319   0.1047    0.0568  0.0105
model 2  0.0141   0.0367   0.1107    0.0531  0.0141
model 3  0.0105   0.0325   0.1152    0.0537  0.0126
Spearman: mean 0.950, min 0.900
Top-2 agreement: 1.00 of 6 pairs
Local cosine: median 0.9916, 5th percentile 0.9016
Total |SHAP difference|: median 0.0271, 95th percentile 0.1138
         past load  calendar  temperature
Jan–Jun     0.1424    0.0360       0.0107
Mar–Aug     0.1437    0.0339       0.0107
May–Oct     0.1473    0.0371       0.0149
Jul–Dec     0.1519    0.0318       0.0128


In [11]:
model = models[0]
phi0 = phis[0]                      # seed-0 SHAP values on X_explain, from Cell 6
pred0 = model.predict(pd.DataFrame(X_explain, columns=FEATS))
q = train[FEATS].quantile([0.25, 0.75])
spread = ((q.loc[0.75] - q.loc[0.25]) / 1.349).to_numpy()
print("robust spread:", dict(zip(FEATS, spread.round(3))))

cont = [2, 3, 4]                    # lag24_n, lag168_n, temp
rng_n = np.random.default_rng(2)
rows = []
for level in [0.01, 0.05, 0.10]:
    Xn = X_explain.copy()
    Xn[:, cont] += rng_n.normal(0, 1, (len(Xn), len(cont))) * level * spread[cont]
    phin, _ = exact_shap(model, Xn, background)
    predn = model.predict(pd.DataFrame(Xn, columns=FEATS))
    d_pred = np.abs(predn - pred0)
    d_shap = np.abs(phin - phi0).sum(axis=1)
    hidden = (d_pred < 0.005) & (d_shap > 0.02)
    rows.append({"noise": level,
                 "median |Δ forecast|": np.median(d_pred),
                 "median total |Δ SHAP|": np.median(d_shap),
                 "p95 total |Δ SHAP|": np.percentile(d_shap, 95),
                 "hidden reshuffles": hidden.mean()})
    print("noise", level, "done", flush=True)
print(pd.DataFrame(rows).round(4).to_string(index=False))

robust spread: {'hour': np.float64(8.895), 'weekday': np.float64(2.965), 'lag24_n': np.float64(0.2), 'lag168_n': np.float64(0.198), 'temp': np.float64(11.119)}
noise 0.01 done
noise 0.05 done
noise 0.1 done
 noise  median |Δ forecast|  median total |Δ SHAP|  p95 total |Δ SHAP|  hidden reshuffles
  0.01               0.0000                 0.0004              0.0123              0.005
  0.05               0.0040                 0.0084              0.0272              0.012
  0.10               0.0094                 0.0167              0.0453              0.042
